In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your actual folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_I.xpt"
csv_file = DATA_DIR / "SMQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 SMQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMQ078",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ621",
    "SMD630",
    "SMQ661",
    "SMQ665A",
    "SMQ665B",
    "SMQ665C",
    "SMQ665D",
    "SMQ670",
    "SMQ848",
    "SMQ852Q",
    "SMQ852U",
    "SMQ925",
    "SMQ930",
    "SMQ935",
    "SMQ080",
    "SMQ890",
    "SMQ895",
    "SMQ900",
    "SMQ905",
    "SMQ910",
    "SMQ915",
    "SMAQUEX2"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQ_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 7001:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 42:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,001 rows x 42 columns."
)


# ============================================================
# 4. DEFINE STRING / CONTINUOUS / INTEGER FIELDS
# ============================================================

#
# SMQ_I contains:
#
# CHARACTER:
#
#     SMDUPCA
#     SMD100BR
#
#
# CONTINUOUS CIGARETTE CONTENT MEASUREMENTS:
#
#     SMD100TR   Tar
#     SMD100NI   Nicotine
#     SMD100CO   Carbon monoxide
#
#
# All remaining numeric fields are proven whole-number
# identifiers / codes / counts.
#
#
# IMPORTANT:
#
# SMDUPCA must NEVER be converted to numeric.
#
# Many UPCs begin with zero.
#

STRING_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


CONTINUOUS_COLUMNS = [
    "SMD100TR",
    "SMD100NI",
    "SMD100CO"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if (
        col not in STRING_COLUMNS
        and
        col not in CONTINUOUS_COLUMNS
    )
]


if len(STRING_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly 2 character fields."
    )


if len(CONTINUOUS_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 continuous fields."
    )


if len(INTEGER_COLUMNS) != 37:

    raise ValueError(
        f"Expected 37 integer/code fields, "
        f"found {len(INTEGER_COLUMNS)}."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Continuous fields:",
    CONTINUOUS_COLUMNS
)

print(
    "Whole-number numeric fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. VALIDATE CHARACTER FIELDS
# ============================================================

#
# IMPORTANT FIX:
#
# DO NOT require:
#
#     df[col].dtype == object
#
# Newer pandas versions may use:
#
#     object
#     string
#     str
#
# All are acceptable.
#
# What matters:
#
#     populated values must actually be strings.
#

print(
    "\n--- CHARACTER FIELD DTYPE CHECK ---"
)


for col in STRING_COLUMNS:

    print(
        f"{col}: dtype = {df[col].dtype}"
    )


    nonmissing = (
        df[col]
        .dropna()
    )


    non_string_count = int(
        (
            ~nonmissing.map(
                lambda x: isinstance(x, str)
            )
        )
        .sum()
    )


    if non_string_count != 0:

        raise ValueError(
            f"{col} contains "
            f"{non_string_count:,} non-string values."
        )


print(
    "PASS: SMDUPCA and SMD100BR contain "
    "character/string values."
)


# ============================================================
# 6. NORMALIZED CHARACTER VIEWS FOR VALIDATION
# ============================================================

#
# We do NOT modify the dataframe.
#
# These temporary Series simply treat missing character
# values as blank strings for validation.
#

upc = (
    df["SMDUPCA"]
    .fillna("")
    .astype(str)
)


brand = (
    df["SMD100BR"]
    .fillna("")
    .astype(str)
)


# ============================================================
# 7. CHECK SMDUPCA CHARACTER STRUCTURE
# ============================================================

#
# CDC defines SMDUPCA as character $12.
#
# A UPC such as:
#
#     026100005738
#
# MUST NOT become:
#
#     26100005738
#

upc_digits = (
    upc
    .str.fullmatch(
        r"\d{12}"
    )
    .fillna(False)
)


upc_no_match = (
    upc
    .eq("* NO MATCH *")
    .fillna(False)
)


upc_blank = (
    upc
    .eq("")
    .fillna(False)
)


upc_recorded_count = int(
    upc_digits.sum()
)


upc_no_match_count = int(
    upc_no_match.sum()
)


upc_blank_count = int(
    upc_blank.sum()
)


upc_leading_zero_count = int(
    (
        upc_digits
        &
        upc.str.startswith("0")
        .fillna(False)
    )
    .sum()
)


print(
    "\n--- SMDUPCA CHARACTER CHECK ---"
)

print(
    "12-digit UPC values:",
    f"{upc_recorded_count:,}"
)

print(
    "* NO MATCH *:",
    f"{upc_no_match_count:,}"
)

print(
    "Blank:",
    f"{upc_blank_count:,}"
)

print(
    "12-digit UPCs beginning with zero:",
    f"{upc_leading_zero_count:,}"
)


if upc_recorded_count != 695:

    raise ValueError(
        "Unexpected SMDUPCA 12-digit UPC count."
    )


if upc_no_match_count != 69:

    raise ValueError(
        "Unexpected SMDUPCA '* NO MATCH *' count."
    )


if upc_blank_count != 6237:

    raise ValueError(
        "Unexpected SMDUPCA blank count."
    )


if upc_leading_zero_count != 652:

    raise ValueError(
        "Unexpected leading-zero UPC count."
    )


if (
    upc_recorded_count
    +
    upc_no_match_count
    +
    upc_blank_count
) != 7001:

    raise ValueError(
        "SMDUPCA categories do not reconcile "
        "to 7,001 rows."
    )


print(
    "PASS: SMDUPCA character structure validated."
)


# ============================================================
# 8. CHECK SMD100BR CHARACTER STRUCTURE
# ============================================================

brand_blank_count = int(
    brand.eq("").sum()
)


brand_nonblank_count = int(
    brand.ne("").sum()
)


print(
    "\n--- SMD100BR CHARACTER CHECK ---"
)

print(
    "Nonblank brand/sub-brand:",
    f"{brand_nonblank_count:,}"
)

print(
    "Blank:",
    f"{brand_blank_count:,}"
)


if brand_nonblank_count != 1004:

    raise ValueError(
        "Unexpected SMD100BR nonblank count."
    )


if brand_blank_count != 5997:

    raise ValueError(
        "Unexpected SMD100BR blank count."
    )


print(
    "PASS: SMD100BR character structure validated."
)


# ============================================================
# 9. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero representation:
#
#     5.397605346934028e-79
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


NUMERIC_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in STRING_COLUMNS
]


actual_numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if actual_numeric_cols != NUMERIC_COLUMNS:

    raise ValueError(
        "Unexpected numeric-column structure.\n\n"
        f"Expected:\n{NUMERIC_COLUMNS}\n\n"
        f"Actual:\n{actual_numeric_cols}"
    )


tiny_counts = (
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[NUMERIC_COLUMNS]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{ordinary_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 10. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached SMQ_I.xpt:
#
#     SMD030       77
#     SMD641       51
#     SMD100FL      9
#     SMD100MN    575
#     SMQ852Q       5
#     SMQ080        9
#     SMQ895     1690
#     SMQ905      768
#     SMQ915      693
#
# TOTAL          3,877
#

EXPECTED_TINY_COUNTS = {
    "SMD030": 77,
    "SMD641": 51,
    "SMD100FL": 9,
    "SMD100MN": 575,
    "SMQ852Q": 5,
    "SMQ080": 9,
    "SMQ895": 1690,
    "SMQ905": 768,
    "SMQ915": 693
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated SMQ_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3877:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 9:

    raise ValueError(
        "Expected tiny artifacts in exactly 9 variables."
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value correction."
    )


print(
    "\nPASS: Exact 3,877-value tiny-zero pattern "
    "across 9 variables validated."
)


# ============================================================
# 11. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT globally do:
#
#     df.replace(TINY_VALUE, 0)
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 3877:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 12. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 13. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 14. DETECT ACTUAL FRACTIONAL VARIABLES
# ============================================================

#
# np.round() is used ONLY for diagnostic comparison.
#
# NO source data is changed.
#

fractional_columns = []

fractional_counts = {}


for col in NUMERIC_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    if count > 0:

        fractional_columns.append(
            col
        )


print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)

print(
    "Variables with actual fractional values:",
    fractional_columns
)


if fractional_columns != ["SMD100NI"]:

    raise ValueError(
        "Unexpected fractional-variable pattern.\n\n"
        "Expected:\n['SMD100NI']\n\n"
        f"Actual:\n{fractional_columns}"
    )


if fractional_counts["SMD100NI"] != 584:

    raise ValueError(
        "Unexpected SMD100NI fractional count."
    )


print(
    "PASS: SMD100NI is the only variable "
    "with actual fractional observations."
)


# ============================================================
# 15. CONTINUOUS MEASUREMENT CHECK
# ============================================================

#
# Preserve ALL THREE as continuous measurements:
#
#     SMD100TR
#     SMD100NI
#     SMD100CO
#
# Even though TR and CO happen to contain whole-looking
# released values in this cycle.
#

EXPECTED_CONTINUOUS = {

    "SMD100TR": {
        "nonmissing": 752,
        "missing": 6249,
        "min": 6.0,
        "max": 24.0,
        "fractional": 0
    },

    "SMD100NI": {
        "nonmissing": 752,
        "missing": 6249,
        "min": 0.5,
        "max": 2.0,
        "fractional": 584
    },

    "SMD100CO": {
        "nonmissing": 752,
        "missing": 6249,
        "min": 1.0,
        "max": 19.0,
        "fractional": 0
    }
}


print(
    "\n--- CONTINUOUS MEASUREMENT CHECK ---"
)


for col, expected in EXPECTED_CONTINUOUS.items():

    s = df[col]


    actual_nonmissing = int(
        s.notna().sum()
    )


    actual_missing = int(
        s.isna().sum()
    )


    actual_min = float(
        s.min()
    )


    actual_max = float(
        s.max()
    )


    actual_fractional = (
        fractional_counts[col]
    )


    print(
        f"\n{col}"
    )

    print(
        "  Nonmissing:",
        actual_nonmissing
    )

    print(
        "  Missing:",
        actual_missing
    )

    print(
        "  Min:",
        repr(actual_min)
    )

    print(
        "  Max:",
        repr(actual_max)
    )

    print(
        "  Fractional:",
        actual_fractional
    )


    if actual_nonmissing != expected["nonmissing"]:

        raise ValueError(
            f"{col} nonmissing count differs."
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs."
        )


    if actual_min != expected["min"]:

        raise ValueError(
            f"{col} minimum differs."
        )


    if actual_max != expected["max"]:

        raise ValueError(
            f"{col} maximum differs."
        )


    if actual_fractional != expected["fractional"]:

        raise ValueError(
            f"{col} fractional count differs."
        )


print(
    "\nPASS: FTC tar/nicotine/CO "
    "measurement structure validated."
)


# ============================================================
# 16. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 17. PROVE 37 INTEGER/CODE FIELDS ARE WHOLE-NUMBER
# ============================================================

integer_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole.all():

        integer_failures.append(
            col
        )


if integer_failures:

    raise ValueError(
        "Unexpected fractional integer/code fields:\n"
        + str(
            integer_failures
        )
    )


print(
    "PASS: All 37 ID/code/count fields "
    "are mathematically whole-number."
)


# ============================================================
# 18. CONVERT ONLY PROVEN INTEGER FIELDS TO Int64
# ============================================================

#
# NO .round() during conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: 37 whole-number fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 19. CONFIRM CONTINUOUS FIELDS REMAIN FLOAT
# ============================================================

for col in CONTINUOUS_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        raise ValueError(
            f"{col} is no longer floating-point."
        )


print(
    "PASS: SMD100TR, SMD100NI and SMD100CO "
    "remain float."
)


# ============================================================
# 20. CONFIRM CHARACTER FIELDS REMAIN CHARACTER
# ============================================================

#
# UPDATED:
#
# Do NOT require dtype == object.
#

print(
    "\n--- CHARACTER FIELDS AFTER NUMERIC CLEANING ---"
)


for col in STRING_COLUMNS:

    nonmissing = (
        df[col]
        .dropna()
    )


    non_string_count = int(
        (
            ~nonmissing.map(
                lambda x: isinstance(x, str)
            )
        )
        .sum()
    )


    print(
        f"{col}: dtype = {df[col].dtype}, "
        f"non-string values = {non_string_count:,}"
    )


    if non_string_count != 0:

        raise ValueError(
            f"{col} contains non-string values "
            "after numeric cleaning."
        )


print(
    "PASS: SMDUPCA and SMD100BR remain "
    "character/string fields."
)


# ============================================================
# 21. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches expected frequencies."
    )


# ============================================================
# 22. CDC CHECK - SMQ020
# ============================================================

check_frequency(
    "SMQ020",
    {
        1: 2422,
        2: 3559,
        7: 2,
        9: 9
    },
    1009
)


# ============================================================
# 23. CDC CHECK - SMD030
# ============================================================

#
# Age first smoked regularly:
#
#     7-58 = 2,328
#     0    =    77 Never smoked regularly
#     777  =     2
#     999  =    15
#
# Missing = 4,579
#

if int(
    df["SMD030"]
    .between(
        7,
        58
    )
    .sum()
) != 2328:

    raise ValueError(
        "SMD030 7-58 count differs."
    )


if int(
    df["SMD030"]
    .eq(0)
    .sum()
) != 77:

    raise ValueError(
        "SMD030 zero count differs."
    )


if int(
    df["SMD030"]
    .eq(777)
    .sum()
) != 2:

    raise ValueError(
        "SMD030 refused count differs."
    )


if int(
    df["SMD030"]
    .eq(999)
    .sum()
) != 15:

    raise ValueError(
        "SMD030 don't-know count differs."
    )


if int(
    df["SMD030"]
    .isna()
    .sum()
) != 4579:

    raise ValueError(
        "SMD030 missing count differs."
    )


print(
    "PASS: SMD030 matches CDC."
)


# ============================================================
# 24. CDC CHECK - SMQ040
# ============================================================

check_frequency(
    "SMQ040",
    {
        1: 832,
        2: 268,
        3: 1322
    },
    4579
)


# ============================================================
# 25. SMQ020 -> SMQ040 ROUTING
# ============================================================

smq020_yes = (
    df["SMQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smq040_present = (
    df["SMQ040"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smq040_mismatch = int(
    np.sum(
        smq020_yes
        !=
        smq040_present
    )
)


if smq040_mismatch != 0:

    raise ValueError(
        "SMQ040 availability differs from SMQ020=1."
    )


print(
    "PASS: SMQ020/SMQ040 routing validated."
)


# ============================================================
# 26. SMD641 ZERO / RANGE CHECK
# ============================================================

#
# Days smoked in past 30 days.
#
# CDC grouped valid range:
#
#     0-30 = 1,160
#     99   = 2
#
# Exact XPT zero count after repair:
#
#     51
#

if int(
    df["SMD641"]
    .between(
        0,
        30
    )
    .sum()
) != 1160:

    raise ValueError(
        "SMD641 0-30 count differs."
    )


if int(
    df["SMD641"]
    .eq(0)
    .sum()
) != 51:

    raise ValueError(
        "SMD641 zero count differs."
    )


if int(
    df["SMD641"]
    .eq(99)
    .sum()
) != 2:

    raise ValueError(
        "SMD641 don't-know count differs."
    )


if int(
    df["SMD641"]
    .isna()
    .sum()
) != 5839:

    raise ValueError(
        "SMD641 missing count differs."
    )


print(
    "PASS: SMD641 validated."
)


# ============================================================
# 27. CDC CHECK - SMD093
# ============================================================

check_frequency(
    "SMD093",
    {
        1: 768,
        2: 241,
        3: 45,
        4: 43,
        7: 3
    },
    5901
)


# ============================================================
# 28. UPC PACK-SHOWING RELATIONSHIP
# ============================================================

upc_nonblank = (
    df["SMDUPCA"]
    .fillna("")
    .astype(str)
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


pack_shown = (
    df["SMD093"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


upc_outside_pack_shown = int(
    np.sum(
        upc_nonblank
        &
        ~pack_shown
    )
)


if int(
    upc_nonblank.sum()
) != 764:

    raise ValueError(
        "Unexpected SMDUPCA nonblank total."
    )


if upc_outside_pack_shown != 0:

    raise ValueError(
        "SMDUPCA populated outside SMD093=1."
    )


print(
    "PASS: SMDUPCA pack-showing relationship validated."
)


# ============================================================
# 29. FILTER / MENTHOL / LENGTH CHECKS
# ============================================================

check_frequency(
    "SMD100FL",
    {
        0: 9,
        1: 995
    },
    5997
)


check_frequency(
    "SMD100MN",
    {
        0: 575,
        1: 429
    },
    5997
)


check_frequency(
    "SMD100LN",
    {
        1: 67,
        2: 538,
        3: 388,
        4: 11
    },
    5997
)


# ============================================================
# 30. SMQ852Q ZERO / RANGE CHECK
# ============================================================

if int(
    df["SMQ852Q"]
    .between(
        0,
        480
    )
    .sum()
) != 594:

    raise ValueError(
        "SMQ852Q 0-480 count differs."
    )


if int(
    df["SMQ852Q"]
    .eq(0)
    .sum()
) != 5:

    raise ValueError(
        "SMQ852Q zero count differs."
    )


if int(
    df["SMQ852Q"]
    .eq(9999)
    .sum()
) != 2:

    raise ValueError(
        "SMQ852Q don't-know count differs."
    )


if int(
    df["SMQ852Q"]
    .isna()
    .sum()
) != 6405:

    raise ValueError(
        "SMQ852Q missing count differs."
    )


print(
    "PASS: SMQ852Q validated."
)


# ============================================================
# 31. SMQ925 ROUTING CHECK
# ============================================================

smq925_expected = (
    df["SMQ020"]
    .isin(
        [2, 7, 9]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smq925_present = (
    df["SMQ925"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smq925_mismatch = int(
    np.sum(
        smq925_expected
        !=
        smq925_present
    )
)


if int(
    smq925_expected.sum()
) != 3570:

    raise ValueError(
        "Unexpected SMQ925 eligibility count."
    )


if smq925_mismatch != 0:

    raise ValueError(
        "SMQ925 availability differs from SMQ020."
    )


print(
    "PASS: SMQ020/SMQ925 routing validated."
)


# ============================================================
# 32. SMQ080 ZERO / RANGE CHECK
# ============================================================

if int(
    df["SMQ080"]
    .between(
        0,
        30
    )
    .sum()
) != 62:

    raise ValueError(
        "SMQ080 0-30 count differs."
    )


if int(
    df["SMQ080"]
    .eq(0)
    .sum()
) != 9:

    raise ValueError(
        "SMQ080 zero count differs."
    )


if int(
    df["SMQ080"]
    .isna()
    .sum()
) != 6939:

    raise ValueError(
        "SMQ080 missing count differs."
    )


print(
    "PASS: SMQ080 validated."
)


# ============================================================
# 33. ADULT TOBACCO QUESTION CHECKS
# ============================================================

check_frequency(
    "SMQ890",
    {
        1: 2066,
        2: 3917,
        7: 3,
        9: 6
    },
    1009
)


check_frequency(
    "SMQ900",
    {
        1: 1064,
        2: 4925,
        7: 2,
        9: 1
    },
    1009
)


check_frequency(
    "SMQ910",
    {
        1: 827,
        2: 5160,
        7: 2,
        9: 3
    },
    1009
)


# ============================================================
# 34. SMQ895 ZERO CHECK
# ============================================================

if int(
    df["SMQ895"]
    .between(
        0,
        30
    )
    .sum()
) != 2065:

    raise ValueError(
        "SMQ895 0-30 count differs."
    )


if int(
    df["SMQ895"]
    .eq(0)
    .sum()
) != 1690:

    raise ValueError(
        "SMQ895 zero count differs."
    )


if int(
    df["SMQ895"]
    .eq(99)
    .sum()
) != 1:

    raise ValueError(
        "SMQ895 don't-know count differs."
    )


if int(
    df["SMQ895"]
    .isna()
    .sum()
) != 4935:

    raise ValueError(
        "SMQ895 missing count differs."
    )


print(
    "PASS: SMQ895 validated."
)


# ============================================================
# 35. SMQ905 ZERO CHECK
# ============================================================

if int(
    df["SMQ905"]
    .between(
        0,
        30
    )
    .sum()
) != 1063:

    raise ValueError(
        "SMQ905 0-30 count differs."
    )


if int(
    df["SMQ905"]
    .eq(0)
    .sum()
) != 768:

    raise ValueError(
        "SMQ905 zero count differs."
    )


if int(
    df["SMQ905"]
    .eq(99)
    .sum()
) != 1:

    raise ValueError(
        "SMQ905 don't-know count differs."
    )


if int(
    df["SMQ905"]
    .isna()
    .sum()
) != 5937:

    raise ValueError(
        "SMQ905 missing count differs."
    )


print(
    "PASS: SMQ905 validated."
)


# ============================================================
# 36. SMQ915 ZERO CHECK
# ============================================================

if int(
    df["SMQ915"]
    .between(
        0,
        30
    )
    .sum()
) != 826:

    raise ValueError(
        "SMQ915 0-30 count differs."
    )


if int(
    df["SMQ915"]
    .eq(0)
    .sum()
) != 693:

    raise ValueError(
        "SMQ915 zero count differs."
    )


if int(
    df["SMQ915"]
    .eq(99)
    .sum()
) != 1:

    raise ValueError(
        "SMQ915 don't-know count differs."
    )


if int(
    df["SMQ915"]
    .isna()
    .sum()
) != 6174:

    raise ValueError(
        "SMQ915 missing count differs."
    )


print(
    "PASS: SMQ915 validated."
)


# ============================================================
# 37. QUESTIONNAIRE MODE CHECK
# ============================================================

check_frequency(
    "SMAQUEX2",
    {
        1: 5992,
        2: 1009
    },
    0
)


adult_mode = (
    df["SMAQUEX2"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smq020_present = (
    df["SMQ020"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


mode_mismatch = int(
    np.sum(
        adult_mode
        !=
        smq020_present
    )
)


if mode_mismatch != 0:

    raise ValueError(
        "SMQ020 availability does not match "
        "SMAQUEX2 interview mode."
    )


print(
    "PASS: Adult/youth questionnaire structure validated."
)


# ============================================================
# 38. FINAL VALIDATED ZERO PATTERN
# ============================================================

final_zero_counts = {}


for col in NUMERIC_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 3877:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the 3,877 validated restorations."
)


# ============================================================
# 39. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

pre_export_numeric_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_numeric_failures.append(
            col
        )


if pre_export_numeric_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_numeric_failures
        )
    )


print(
    "PASS: All numeric values match "
    "the corrected XPT exactly."
)


# ============================================================
# 40. CHARACTER FIDELITY BEFORE EXPORT
# ============================================================

for col in STRING_COLUMNS:

    expected_values = (
        expected_df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    current_values = (
        df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    mismatches = int(
        np.sum(
            expected_values
            !=
            current_values
        )
    )


    if mismatches != 0:

        raise ValueError(
            f"{col}: {mismatches:,} character values "
            "changed before export."
        )


print(
    "PASS: SMDUPCA and SMD100BR remain "
    "character-for-character unchanged."
)


# ============================================================
# 41. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# This preserves:
#
#     genuine decimals
#     measurement values
#     leading-zero UPC values
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 42. RE-READ CSV FOR NUMERIC FIDELITY
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 43. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

numeric_csv_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        numeric_csv_failures.append(
            col
        )


if numeric_csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            numeric_csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all numeric SMQ_I fields."
)


# ============================================================
# 44. NUMERIC MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0


for col in NUMERIC_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nNumeric missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Numeric missing-value positions changed."
    )


print(
    "PASS: Numeric missing positions preserved exactly."
)


# ============================================================
# 45. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Essential for:
#
#     leading-zero UPC validation
#     exact brand validation
#     .0 checks
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 46. EXACT SMDUPCA ROUND-TRIP FIDELITY
# ============================================================

expected_upc = (
    expected_df["SMDUPCA"]
    .fillna("")
    .astype(str)
    .to_numpy()
)


exported_upc = (
    raw_csv_df["SMDUPCA"]
    .to_numpy()
)


upc_mismatches = int(
    np.sum(
        expected_upc
        !=
        exported_upc
    )
)


print(
    "\n--- SMDUPCA ROUND-TRIP ---"
)

print(
    "Character mismatches:",
    upc_mismatches
)


if upc_mismatches != 0:

    raise ValueError(
        "SMDUPCA changed during CSV export."
    )


final_upc_digits = (
    raw_csv_df["SMDUPCA"]
    .str.fullmatch(
        r"\d{12}"
    )
    .fillna(False)
)


final_leading_zero_count = int(
    (
        final_upc_digits
        &
        raw_csv_df["SMDUPCA"]
        .str.startswith("0")
        .fillna(False)
    )
    .sum()
)


if int(
    final_upc_digits.sum()
) != 695:

    raise ValueError(
        "Final 12-digit UPC count changed."
    )


if int(
    raw_csv_df["SMDUPCA"]
    .eq("* NO MATCH *")
    .sum()
) != 69:

    raise ValueError(
        "Final UPC no-match count changed."
    )


if final_leading_zero_count != 652:

    raise ValueError(
        "Final leading-zero UPC count changed."
    )


print(
    "PASS: All UPC values preserved, "
    "including 652 leading-zero UPCs."
)


# ============================================================
# 47. EXACT SMD100BR ROUND-TRIP FIDELITY
# ============================================================

expected_brand = (
    expected_df["SMD100BR"]
    .fillna("")
    .astype(str)
    .to_numpy()
)


exported_brand = (
    raw_csv_df["SMD100BR"]
    .to_numpy()
)


brand_mismatches = int(
    np.sum(
        expected_brand
        !=
        exported_brand
    )
)


print(
    "\n--- SMD100BR ROUND-TRIP ---"
)

print(
    "Character mismatches:",
    brand_mismatches
)


if brand_mismatches != 0:

    raise ValueError(
        "SMD100BR changed during CSV export."
    )


if int(
    raw_csv_df["SMD100BR"]
    .ne("")
    .sum()
) != 1004:

    raise ValueError(
        "Final brand nonblank count changed."
    )


print(
    "PASS: SMD100BR preserved exactly."
)


# ============================================================
# 48. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 49. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally check:
#
#     SMD100TR
#     SMD100NI
#     SMD100CO
#
# because they are deliberately preserved as
# continuous measurement columns.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer/code fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 37 whole-number fields."
)


# ============================================================
# 50. LEGITIMATE CONTINUOUS .0 CHECK
# ============================================================

#
# Expected test-export pattern:
#
# SMD100TR:
#     752 .0 tokens
#
# SMD100NI:
#     168 .0 tokens
#
# SMD100CO:
#     752 .0 tokens
#
#
# These are NOT fidelity errors.
#

EXPECTED_CONTINUOUS_DOT_ZERO = {
    "SMD100TR": 752,
    "SMD100NI": 168,
    "SMD100CO": 752
}


actual_continuous_dot_zero = {}


for col in CONTINUOUS_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    actual_continuous_dot_zero[col] = count


print(
    "\n--- LEGITIMATE CONTINUOUS .0 TOKENS ---"
)

print(
    actual_continuous_dot_zero
)


if (
    actual_continuous_dot_zero
    !=
    EXPECTED_CONTINUOUS_DOT_ZERO
):

    raise ValueError(
        "Continuous '.0' token pattern changed."
    )


print(
    "PASS: Continuous measurement formatting preserved."
)


# ============================================================
# 51. EXACT CONTINUOUS NUMERIC FIDELITY
# ============================================================

continuous_failures = []


for col in CONTINUOUS_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            raw_csv_df[col]
            .where(
                raw_csv_df[col] != "",
                np.nan
            ),
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        continuous_failures.append(
            col
        )


if continuous_failures:

    raise ValueError(
        "Continuous measurement fidelity failed:\n"
        + str(
            continuous_failures
        )
    )


print(
    "PASS: SMD100TR, SMD100NI and SMD100CO "
    "preserved numerically exactly."
)


# ============================================================
# 52. FINAL ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 3,877 validated zero values."
)


# ============================================================
# 53. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in NUMERIC_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 54. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 55. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "SMQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Smoking/Cigarette Use"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Character fields:",
    STRING_COLUMNS
)


print(
    "Continuous measurement fields:",
    CONTINUOUS_COLUMNS
)


print(
    "Actual fractional field:",
    "SMD100NI"
)


print(
    "SMD100NI fractional observations:",
    fractional_counts["SMD100NI"]
)


print(
    "Whole-number numeric fields:",
    len(INTEGER_COLUMNS)
)


print(
    "12-digit UPC records:",
    upc_recorded_count
)


print(
    "Leading-zero UPC records:",
    upc_leading_zero_count
)


print(
    "UPC no-match records:",
    upc_no_match_count
)


print(
    "Brand/sub-brand records:",
    brand_nonblank_count
)


print(
    "Numeric XPT -> CSV differences:",
    "0 expected"
)


print(
    "UPC character differences:",
    "0 expected"
)


print(
    "Brand character differences:",
    "0 expected"
)


print(
    "Numeric missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)


print(
    "Continuous measurement '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "Only the 3,877 validated tiny XPORT "
    "zero artifacts are repaired."
)


print(
    "SMDUPCA and SMD100BR remain exact "
    "character fields."
)


print(
    "SMD100TR, SMD100NI and SMD100CO remain "
    "continuous measurement fields."
)


print(
    "Only the 37 proven whole-number numeric fields "
    "are converted to nullable Int64."
)


print(
    "No genuine continuous value is rounded."
)


print(
    "No float_format is used during CSV export."
)


print(
    "Leading-zero UPC values remain intact."
)

NHANES 2015-2016 SMQ_I XPT -> CSV
Rows: 7,001
Columns: 42
PASS: Structure = 7,001 rows x 42 columns.

--- FIELD TYPE PLAN ---
Character fields: ['SMDUPCA', 'SMD100BR']
Continuous fields: ['SMD100TR', 'SMD100NI', 'SMD100CO']
Whole-number numeric fields: 37

--- CHARACTER FIELD DTYPE CHECK ---
SMDUPCA: dtype = str
SMD100BR: dtype = str
PASS: SMDUPCA and SMD100BR contain character/string values.

--- SMDUPCA CHARACTER CHECK ---
12-digit UPC values: 695
* NO MATCH *: 69
Blank: 6,237
12-digit UPCs beginning with zero: 652
PASS: SMDUPCA character structure validated.

--- SMD100BR CHARACTER CHECK ---
Nonblank brand/sub-brand: 1,004
Blank: 5,997
PASS: SMD100BR character structure validated.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 3,877
Variables containing tiny artifacts: 9

Tiny values by variable:
SMQ895      1690
SMQ905       768
SMQ915       693
SMD100MN     575
SMD030        77
SMD641        51
SMD100FL       9
SMQ080         9
S